# V030: one fixed cross-objective hybrid mixture
This separate prospective screen splits the original curve-model slot equally among plain, span3, span7 and Laplace models. One candidate only, no GPU, no weight search. Prior batches remain rejected; repeated-A adaptivity and cold grouped resampling limits are explicit in the frozen protocol.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, time
import numpy as np
import pandas as pd

def v30_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def v30_code(p):
    return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
def v30_require(c,m):
    if not bool(c):raise ValueError(m)
def run_mixture_screen_v030(root, *, expected_protocol_sha256, expected_source_code_sha256):
    root=Path(root).resolve();out=root/'outputs/mixture_screen_v030';started=time.monotonic()
    v30_require(not (out/'results.json').exists(),'Preserve completed results; use a new experiment for new recipes')
    path=root/'research/protocol_v030.json'
    v30_require(v30_sha(path)==expected_protocol_sha256,'Protocol changed')
    source_sha=hashlib.sha256(v30_code(root/'research/mixture_screen_v030.ipynb').encode()).hexdigest()
    v30_require(source_sha==expected_source_code_sha256,'Source changed')
    plan=json.loads(path.read_text())
    for name,h in plan['input_hashes'].items():v30_require(v30_sha(root/name)==h,'Input changed: '+name)
    ns={}
    for name,h in plan['component_code_hashes'].items():
        s=v30_code(root/name);v30_require(hashlib.sha256(s.encode()).hexdigest()==h,'Component changed: '+name);exec(compile(s,name,'exec'),ns)
    audit=json.loads((root/'research/screen_review_v024.json').read_text())
    v30_require(audit['status']=='passed' and all(audit['checks'].values()) and audit['selected'] is None,'V024 audit/promotion state changed')
    v30_require(audit['screen_summary_sha256']==v30_sha(root/'outputs/controlled_experiments_v024/screen_summary.json'),'Audit not bound to current V024')
    laplace_audit=json.loads((root/'research/screen_review_v029.json').read_text());v30_require(laplace_audit['status']=='passed' and all(laplace_audit['checks'].values()) and laplace_audit['selected'] is None and laplace_audit['screen_summary_sha256']==v30_sha(root/'outputs/controlled_experiments_v029/screen_summary.json'),'V029 audit not current or incorrect prior promotion state')
    frame=pd.read_csv(root/'data/train.csv',usecols=['id','breath_id','u_out','pressure'],dtype={'id':'int64','breath_id':'int32','u_out':'int8','pressure':'float32'})
    ids=frame.id.to_numpy().reshape(-1,80);bgroup=frame.breath_id.to_numpy().reshape(-1,80)
    y=frame.pressure.to_numpy().reshape(-1,80);phase=frame.u_out.to_numpy().reshape(-1,80);bids=bgroup[:,0]
    v30_require(np.all(bgroup==bids[:,None]) and len(np.unique(bids))==67905,'Complete contiguous unique breaths required')
    with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z:
        fit=np.sort(pd.Index(bids).get_indexer(z['A_fit']));val=np.sort(pd.Index(bids).get_indexer(z['A_validation']))
    v30_require(len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size and np.array_equal(np.sort(np.r_[fit,val]),np.arange(67905)),'Original A split changed')
    mask=phase[val]==0;counts=mask.sum(1);truth=y[val][mask].astype(np.float64);rows=ids[val][mask]
    v30_require(len(rows)==206098 and np.array_equal(rows,np.load(root/'outputs/improvements_v005/sequence/A_validation_row_ids.npy')) and np.array_equal(rows,np.load(root/'outputs/controlled_experiments_v024/A_row_ids.npy')),'Scored A rows changed')
    grid=ns['pressure_codebook_v016'](y[fit],phase[fit])
    v30_require(np.array_equal(grid,np.load(root/'outputs/postprocessing_screen_v016/A_fit_pressure_codebook.npy')),'Fit-only pressure grid changed')
    predictions={k:np.load(root/name,allow_pickle=False) for k,name in plan['prediction_files'].items()}
    v30_require(all(a.dtype==np.float64 and a.shape==truth.shape and np.isfinite(a).all() for a in predictions.values()),'Native model predictions invalid')
    def policy(weights):return ns['selected_postprocessing_v016']({k:predictions[k] for k in weights},weights,grid)
    reference=policy(plan['reference_weights'])
    v30_require(np.array_equal(reference,np.load(root/plan['reference_prediction_file'])) and abs(np.abs(reference-truth).mean()-plan['reference_A_MAE'])<1e-15,'Reference reconstruction differs')
    p=policy(plan['candidate_weights']);error=np.abs(p-truth);old=np.abs(reference-truth);delta=error-old
    sums=np.add.reduceat(delta,np.r_[0,np.cumsum(counts)[:-1]]);rng=np.random.default_rng(plan['bootstrap_seed']);draws=[]
    for _ in range(plan['bootstrap_draws']):
        take=rng.integers(0,len(counts),size=len(counts));draws.append(float(sums[take].sum()/counts[take].sum()))
    lo,hi=np.quantile(draws,[.025,.975]);mae=float(error.mean());gain=float(old.mean()-mae)
    out.mkdir(parents=True,exist_ok=True)
    predpath=out/(plan['candidate_name']+'_A_predictions.npy')
    v30_require(not predpath.exists(),'Preserve candidate artifact')
    with predpath.open('xb') as f:np.save(f,p,allow_pickle=False)
    eligible=bool(gain>=plan['minimum_A_gain'] and hi<0)
    result=dict(status='complete',created_utc=datetime.now(timezone.utc).isoformat(),protocol_sha256=expected_protocol_sha256,source_code_sha256=source_sha,source_fingerprints=plan['component_code_hashes'],reference_A_MAE=float(old.mean()),name=plan['candidate_name'],weights=plan['candidate_weights'],MAE=mae,median_absolute_error=float(np.median(error)),gain=gain,interval=dict(low=float(lo),high=float(hi),seed=plan['bootstrap_seed'],draws=plan['bootstrap_draws']),eligible=eligible,selected_for_confirmation=plan['candidate_name'] if eligible else None,prediction_file=str(predpath.relative_to(root)),prediction_sha256=v30_sha(predpath),row_ids_sha256=hashlib.sha256(np.ascontiguousarray(rows).tobytes()).hexdigest(),validation_breaths=len(val),validation_scored_points=len(rows),grid_sha256=v30_sha(root/'outputs/postprocessing_screen_v016/A_fit_pressure_codebook.npy'),model_fits=0,no_F_fit_performed=True,no_final_fit_performed=True,no_submission=True,official_best_preserved=.17427,runtime_seconds=time.monotonic()-started)
    v30_require(v30_sha(root/plan['preserved_official_CSV'])==plan['preserved_official_CSV_sha256'],'Accepted CSV changed')
    with (out/'results.json').open('x') as f:f.write(json.dumps(result,indent=2,allow_nan=False)+'\n')
    return result

if os.environ.get('PA3_RUN_V030','0')=='1':
    V030_RESULT=run_mixture_screen_v030(Path.cwd(),expected_protocol_sha256=os.environ['PA3_V030_PROTOCOL_SHA256'],expected_source_code_sha256=os.environ['PA3_V030_SOURCE_SHA256'])
    print(json.dumps(V030_RESULT,indent=2))
